In [1]:
!pip uninstall -y torch torchvision numpy opencv-python facenet-pytorch pillow -q


In [2]:
!pip install numpy==1.19.2
!pip install torch==1.8.0 torchvision==0.9.0
!pip install opencv-python==4.5.1.48
!pip install pillow==8.4.0
!pip install facenet-pytorch==2.5.2


  Using cached numpy-1.19.2.zip (7.3 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (pyproject.toml) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.
ERROR: Could not find a version that satisfies the requirement torch==1.8.0 (from versions: 1.13.0, 1.13.1, 2.0.0, 2.0.1, 2.1.0, 2.1.1, 2.1.2, 2.2.0, 2.2.1, 2.2.2, 2.3.0, 2.3.1, 2.4.0, 2.4.1, 2.5.0, 2.5.1, 2.6.0, 2.7.0, 2.7.1, 2.8.0, 2.9.0, 2.9.1)
ERROR: No matching distribution found for torch==1.8.0
  Using cached opencv-python-4.5.1.48.tar.gz (88.3 MB)
  error: su

In [6]:
from facenet_pytorch import MTCNN
from pathlib import Path
import random
import shutil
import torch
import numpy as np
from PIL import Image
from tqdm import tqdm


# BASE MODEL

In [7]:
BASE = Path("/kaggle/input/face-detection-dataset")

train_img = BASE / "images/train"
train_lbl = BASE / "labels/train"

test_img = Path("/kaggle/working/images/test")
test_lbl = Path("/kaggle/working/labels/test")

test_img.mkdir(parents=True, exist_ok=True)
test_lbl.mkdir(parents=True, exist_ok=True)

images = list(train_img.glob("*.jpg"))
random.seed(42)
test_images = random.sample(images, 300)

for img in test_images:
    shutil.copy(img, test_img / img.name)
    lbl = train_lbl / (img.stem + ".txt")
    if lbl.exists():
        shutil.copy(lbl, test_lbl / lbl.name)

print("Test images:", len(list(test_img.glob("*.jpg"))))

Test images: 300


In [8]:
def load_yolo_labels(label_path, img_w, img_h):
    boxes = []
    if not label_path.exists():
        return boxes

    for line in label_path.read_text().splitlines():
        _, x, y, w, h = map(float, line.split())
        x1 = (x - w/2) * img_w
        y1 = (y - h/2) * img_h
        x2 = (x + w/2) * img_w
        y2 = (y + h/2) * img_h
        boxes.append([x1, y1, x2, y2])
    return boxes


In [9]:
device = "cuda" if torch.cuda.is_available() else "cpu"
mtcnn = MTCNN(keep_all=True, device=device)


In [11]:
def iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    inter = max(0, xB-xA) * max(0, yB-yA)
    areaA = (boxA[2]-boxA[0])*(boxA[3]-boxA[1])
    areaB = (boxB[2]-boxB[0])*(boxB[3]-boxB[1])
    return inter / (areaA + areaB - inter + 1e-6)


In [12]:
TP, FP, FN = 0, 0, 0
ious = []

for img_path in tqdm(list(test_img.glob("*.jpg"))):
    img = Image.open(img_path).convert("RGB")
    w, h = img.size

    gt_boxes = load_yolo_labels(test_lbl / (img_path.stem + ".txt"), w, h)
    preds, _ = mtcnn.detect(img)

    matched = set()

    if preds is not None:
        for p in preds:
            best_iou = 0
            best_gt = -1
            for i, g in enumerate(gt_boxes):
                v = iou(p, g)
                if v > best_iou:
                    best_iou, best_gt = v, i
            if best_iou >= 0.5:
                TP += 1
                matched.add(best_gt)
                ious.append(best_iou)
            else:
                FP += 1

    FN += len(gt_boxes) - len(matched)


100%|██████████| 300/300 [00:27<00:00, 11.05it/s]


In [13]:
precision = TP / (TP + FP + 1e-6)
recall = TP / (TP + FN + 1e-6)

mAP50 = np.mean([i for i in ious if i >= 0.5]) if ious else 0
mAP5095 = np.mean(ious) if ious else 0

print("Precision:", precision)
print("Recall:", recall)
print("mAP@0.5:", mAP50)
print("mAP@0.5:0.95:", mAP5095)

Precision: 0.7948113198174395
Recall: 0.7590090081542691
mAP@0.5: 0.7780874
mAP@0.5:0.95: 0.7780874


In [ ]:
# Left blank on purpose